# GAN-Based Automatic Generation of Intermediate Code
**Compiler Design Lab — CSE3024**

Executable prototype based on the report. It demonstrates a restricted source-language parser, AST construction, deterministic three-address-code (TAC) generation, validation, a synthetic paired dataset, and a compact PyTorch conditional generator/discriminator experiment.

**Important:** this is a lab-scale experiment, not a production compiler. Neural output is not guaranteed to be valid or semantically equivalent. The notebook reports measured results when run; it does not assume or fabricate performance.

## 1. Environment

In [ ]:
import sys, re, random, time, platform
from dataclasses import dataclass
import numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
print("Python", sys.version.split()[0], "| NumPy", np.__version__, "|", platform.platform())

## 2. Tokenizer, parser, and AST

In [ ]:
TOKEN_RE = re.compile(r"\s*(?:(\d+)|([A-Za-z_]\w*)|([+\-*/=;(){}])|(.))")
def tokenize(source):
    out=[]; pos=0
    while pos < len(source):
        m=TOKEN_RE.match(source,pos)
        if not m: raise SyntaxError(f"Cannot tokenize at {pos}")
        number, ident, op, bad=m.groups(); pos=m.end()
        if bad: raise SyntaxError(f"Unexpected character {bad!r}")
        if number: out.append(("NUMBER",number))
        elif ident: out.append((ident.upper() if ident in {"int","return"} else "ID",ident))
        elif op: out.append((op,op))
    out.append(("EOF","")); return out

@dataclass
class Node:
    kind: str
    value: object = None
    children: tuple = ()

class Parser:
    def __init__(self,tokens): self.t=tokens; self.i=0
    def peek(self): return self.t[self.i][0]
    def take(self,k=None):
        x=self.t[self.i]
        if k and x[0]!=k: raise SyntaxError(f"Expected {k}, got {x}")
        self.i+=1; return x
    def parse(self):
        ss=[]
        while self.peek()!="EOF": ss.append(self.statement())
        return Node("Program",children=tuple(ss))
    def statement(self):
        if self.peek()=="INT":
            self.take("INT"); name=self.take("ID")[1]; self.take("=")
            e=self.expr(); self.take(";"); return Node("Declare",name,(e,))
        if self.peek()=="RETURN":
            self.take("RETURN"); e=self.expr(); self.take(";"); return Node("Return",children=(e,))
        name=self.take("ID")[1]; self.take("="); e=self.expr(); self.take(";")
        return Node("Assign",name,(e,))
    def expr(self): return self.add()
    def add(self):
        n=self.mul()
        while self.peek() in {"+","-"}:
            op=self.take()[0]; n=Node("BinOp",op,(n,self.mul()))
        return n
    def mul(self):
        n=self.factor()
        while self.peek() in {"*","/"}:
            op=self.take()[0]; n=Node("BinOp",op,(n,self.factor()))
        return n
    def factor(self):
        k,v=self.t[self.i]
        if k=="NUMBER": self.take(); return Node("Number",int(v))
        if k=="ID": self.take(); return Node("Id",v)
        if k=="(": self.take("("); n=self.expr(); self.take(")"); return n
        raise SyntaxError(f"Expected expression, got {k}:{v}")

def parse_source(s): return Parser(tokenize(s)).parse()
def show_ast(n,depth=0):
    label=n.kind if n.value is None else f"{n.kind}({n.value})"
    return "\n".join(["  "*depth+label]+[show_ast(c,depth+1) for c in n.children])

example = "int a = 4; int b = 3; int c = a + b * 2; return c;"
ast=parse_source(example)
print(show_ast(ast))

## 3. Semantic checks and deterministic TAC baseline

In [ ]:
def identifiers(n):
    out=[n.value] if n.kind=="Id" else []
    for c in n.children: out += identifiers(c)
    return out

def semantic_check(ast):
    declared=set()
    for s in ast.children:
        if s.kind in {"Declare","Assign"}:
            if s.kind=="Assign" and s.value not in declared: raise ValueError(f"Undeclared assignment target: {s.value}")
            for x in identifiers(s.children[0]):
                if x not in declared: raise ValueError(f"Use before declaration: {x}")
            if s.kind=="Declare":
                if s.value in declared: raise ValueError(f"Duplicate declaration: {s.value}")
                declared.add(s.value)
        elif s.kind=="Return":
            for x in identifiers(s.children[0]):
                if x not in declared: raise ValueError(f"Use before declaration: {x}")
    return declared

def generate_tac(ast):
    lines=[]; count=[0]
    def expr(n):
        if n.kind=="Number": return str(n.value)
        if n.kind=="Id": return n.value
        if n.kind=="BinOp":
            a=expr(n.children[0]); b=expr(n.children[1]); count[0]+=1
            t=f"t{count[0]}"; lines.append(f"{t} = {a} {n.value} {b}"); return t
        raise ValueError(n.kind)
    for s in ast.children:
        if s.kind in {"Declare","Assign"}: lines.append(f"{s.value} = {expr(s.children[0])}")
        elif s.kind=="Return": lines.append(f"return {expr(s.children[0])}")
    return lines

semantic_check(ast)
reference_tac=generate_tac(ast)
print("\n".join(reference_tac))

## 4. TAC validation and execution comparison

In [ ]:
BIN=re.compile(r"^(t\d+)\s*=\s*(\w+|-?\d+)\s*([+\-*/])\s*(\w+|-?\d+)$")
ASSIGN=re.compile(r"^([A-Za-z_]\w*)\s*=\s*(\w+|-?\d+)$")
RET=re.compile(r"^return\s+(\w+|-?\d+)$")
def validate_tac(lines):
    defined=set(); errors=[]
    for i,line in enumerate(lines,1):
        line=line.strip(); m=BIN.match(line)
        if m:
            dest,a,op,b=m.groups()
            for x in (a,b):
                if x.isidentifier() and x not in defined: errors.append(f"Line {i}: {x} used before definition")
            defined.add(dest); continue
        m=ASSIGN.match(line)
        if m:
            dest,src=m.groups()
            if src.isidentifier() and src not in defined: errors.append(f"Line {i}: {src} used before definition")
            defined.add(dest); continue
        m=RET.match(line)
        if m:
            x=m.group(1)
            if x.isidentifier() and x not in defined: errors.append(f"Line {i}: return uses undefined {x}")
            continue
        errors.append(f"Line {i}: invalid instruction {line!r}")
    if not any(x.startswith("return ") for x in lines): errors.append("Missing return")
    return {"valid":not errors,"errors":errors}

def eval_expr(n,env):
    if n.kind=="Number": return n.value
    if n.kind=="Id": return env[n.value]
    a,b=(eval_expr(c,env) for c in n.children)
    return {"+":lambda:a+b,"-":lambda:a-b,"*":lambda:a*b,"/":lambda:int(a/b)}[n.value]()

def run_source(ast):
    env={}; result=None
    for s in ast.children:
        if s.kind in {"Declare","Assign"}: env[s.value]=eval_expr(s.children[0],env)
        elif s.kind=="Return": result=eval_expr(s.children[0],env)
    return result

def run_tac(lines):
    env={}; result=None
    val=lambda x: int(x) if re.fullmatch(r"-?\d+",x) else env[x]
    for line in lines:
        m=BIN.match(line)
        if m:
            d,a,op,b=m.groups(); x,y=val(a),val(b)
            env[d]={"+":x+y,"-":x-y,"*":x*y,"/":int(x/y)}[op]; continue
        m=ASSIGN.match(line)
        if m:
            d,s=m.groups(); env[d]=val(s); continue
        m=RET.match(line)
        if m: result=val(m.group(1))
    return result

print("Validation:",validate_tac(reference_tac))
print("Source result:",run_source(ast),"| TAC result:",run_tac(reference_tac))
assert validate_tac(reference_tac)["valid"]
assert run_source(ast)==run_tac(reference_tac)

## 5. Create synthetic source–IR pairs

In [ ]:
ops=["+","-","*"]
dataset=[]
for a in range(1,5):
  for b in range(1,4):
    for c in range(1,3):
      for op1 in ops:
        for op2 in ops:
          src=f"int a = {a}; int b = {b}; int c = {c}; int out = (a {op1} b) {op2} c; return out;"
          tree=parse_source(src); semantic_check(tree)
          dataset.append({"source":src,"ast":tree,"condition":" ".join(show_ast(tree).split()),"ir":generate_tac(tree)})
print("Dataset size:",len(dataset))
print("Sample:",dataset[0]["source"],"\n",dataset[0]["ir"])

## 6. Neural experiment setup (PyTorch)

In [ ]:
try:
    import torch
    import torch.nn as nn
except ImportError as e:
    raise ImportError("Install PyTorch in this notebook environment: pip install torch") from e
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
print("PyTorch:",torch.__version__,"| device:",device)

In [ ]:
def words(s): return re.findall(r"[A-Za-z_]\w*|-?\d+|[+*/=;()\-]",s)
special=["<PAD>","<BOS>","<EOS>","<UNK>"]
vocab={x:i for i,x in enumerate(special)}
for ex in dataset:
    for w in words(ex["condition"])+words(" ; ".join(ex["ir"])):
        if w not in vocab: vocab[w]=len(vocab)
ivocab={i:w for w,i in vocab.items()}
PAD,BOS,EOS,UNK=[vocab[x] for x in special]
def encode(s,bos=False,eos=False):
    z=[vocab.get(w,UNK) for w in words(s)]
    return ([BOS]+z if bos else z)+([EOS] if eos else [])
MAX_SRC=max(len(encode(x["condition"])) for x in dataset)
MAX_TGT=max(len(encode(" ; ".join(x["ir"]),True,True)) for x in dataset)
def pad(z,n): return (z+[PAD]*n)[:n]
X=torch.tensor([pad(encode(x["condition"]),MAX_SRC) for x in dataset],dtype=torch.long)
Y=torch.tensor([pad(encode(" ; ".join(x["ir"]),True,True),MAX_TGT) for x in dataset],dtype=torch.long)
perm=np.random.default_rng(SEED).permutation(len(dataset)); cut=int(.8*len(perm))
tr,te=perm[:cut],perm[cut:]
Xtr,Ytr=X[tr].to(device),Y[tr].to(device)
Xte,Yte=X[te].to(device),Y[te].to(device)
print("Vocab:",len(vocab),"| train:",len(tr),"| test:",len(te))

In [ ]:
class Generator(nn.Module):
    def __init__(self,nv,emb=48,hid=64):
        super().__init__()
        self.emb=nn.Embedding(nv,emb,padding_idx=PAD)
        self.enc=nn.GRU(emb,hid,batch_first=True)
        self.dec=nn.GRU(emb+hid,hid,batch_first=True)
        self.out=nn.Linear(hid,nv)
    def forward(self,src,tgt):
        _,h=self.enc(self.emb(src))
        ctx=h[-1].unsqueeze(1).expand(-1,tgt.size(1),-1)
        z,_=self.dec(torch.cat([self.emb(tgt),ctx],dim=-1),h)
        return self.out(z)
    @torch.no_grad()
    def generate(self,src,max_len):
        self.eval(); _,h=self.enc(self.emb(src)); ctx=h[-1].unsqueeze(1)
        tok=torch.full((src.size(0),1),BOS,device=src.device,dtype=torch.long); out=[]
        for _ in range(max_len):
            z,h=self.dec(torch.cat([self.emb(tok),ctx],dim=-1),h)
            tok=self.out(z[:,-1]).argmax(-1,keepdim=True); out.append(tok)
            if (tok==EOS).all(): break
        return torch.cat(out,dim=1)

class Discriminator(nn.Module):
    def __init__(self,nv,emb=24,hid=48):
        super().__init__(); self.emb=nn.Embedding(nv,emb,padding_idx=PAD)
        self.net=nn.Sequential(nn.Linear(2*emb,hid),nn.ReLU(),nn.Linear(hid,1))
    def forward(self,src,ir):
        a=self.emb(src).mean(1); b=self.emb(ir).mean(1)
        return self.net(torch.cat([a,b],dim=1)).squeeze(1)

G=Generator(len(vocab)).to(device); D=Discriminator(len(vocab)).to(device)

### 6.1 Supervised generator pretraining

In [ ]:
EPOCHS=8; BATCH=64
optG=torch.optim.Adam(G.parameters(),lr=0.002)
criterion=nn.CrossEntropyLoss(ignore_index=PAD)
t0=time.time()
for ep in range(EPOCHS):
    G.train(); order=torch.randperm(len(Xtr),device=device); losses=[]
    for j in range(0,len(order),BATCH):
        ix=order[j:j+BATCH]; src,tgt=Xtr[ix],Ytr[ix]
        logits=G(src,tgt[:,:-1])
        loss=criterion(logits.reshape(-1,logits.size(-1)),tgt[:,1:].reshape(-1))
        optG.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(),1.0); optG.step()
        losses.append(loss.item())
    print(f"Epoch {ep+1}/{EPOCHS}: loss={np.mean(losses):.4f}")
print(f"Elapsed: {time.time()-t0:.1f}s")

### 6.2 Discriminator training

In [ ]:
def pad_tensor(t,n):
    if t.size(1)>=n: return t[:,:n]
    return torch.cat([t,torch.full((t.size(0),n-t.size(1)),PAD,device=t.device,dtype=t.dtype)],1)
optD=torch.optim.Adam(D.parameters(),lr=0.001)
bce=nn.BCEWithLogitsLoss()
for ep in range(3):
    D.train(); losses=[]; order=torch.randperm(len(Xtr),device=device)
    for j in range(0,len(order),BATCH):
        ix=order[j:j+BATCH]; src,real=Xtr[ix],Ytr[ix]
        with torch.no_grad(): fake=pad_tensor(G.generate(src,real.size(1)),real.size(1))
        lr=D(src,real); lf=D(src,fake)
        loss=bce(lr,torch.ones_like(lr))+bce(lf,torch.zeros_like(lf))
        optD.zero_grad(); loss.backward(); optD.step(); losses.append(loss.item())
    print(f"Discriminator epoch {ep+1}/3: loss={np.mean(losses):.4f}")
print("Note: this prototype trains D but does not adversarially update G.")

## 7. Generate a candidate

In [ ]:
def decode(seq):
    out=[]
    for i in seq:
        i=int(i)
        if i==EOS: break
        if i not in {PAD,BOS}: out.append(ivocab.get(i,"<UNK>"))
    return " ".join(out)
def predict(source):
    tree=parse_source(source); semantic_check(tree)
    cond=" ".join(show_ast(tree).split())
    src=torch.tensor([pad(encode(cond),MAX_SRC)],dtype=torch.long,device=device)
    return decode(G.generate(src,MAX_TGT)[0].cpu().tolist())
candidate=predict(example)
print("Candidate tokens:\n",candidate)
print("\nReference TAC:\n","\n".join(reference_tac))

## 8. Evaluation and baseline comparison

In [ ]:
def candidate_lines(s):
    # Model output is a token sequence; split on semicolon separators.
    return [x.strip() for x in s.split(";") if x.strip()]
n=min(100,len(te)); valid=exact=semantic=0
for idx in te[:n]:
    ex=dataset[int(idx)]; pred=predict(ex["source"]); lines=candidate_lines(pred)
    report=validate_tac(lines); valid+=int(report["valid"])
    exact+=int(words(pred)==words(" ; ".join(ex["ir"])))
    if report["valid"]:
        try: semantic+=int(run_tac(lines)==run_source(ex["ast"]))
        except Exception: pass
print(f"Neural test examples: {n}")
print(f"Validator pass rate: {valid/n:.1%}")
print(f"Exact token match: {exact/n:.1%}")
print(f"Semantic agreement (finite test set): {semantic/n:.1%}")
bv=bs=0
for idx in te:
    ex=dataset[int(idx)]; tac=generate_tac(ex["ast"])
    bv+=int(validate_tac(tac)["valid"]); bs+=int(run_tac(tac)==run_source(ex["ast"]))
print(f"Rule-based validator pass rate: {bv/len(te):.1%}")
print(f"Rule-based semantic agreement: {bs/len(te):.1%}")

## 9. Limitations and next steps

- Supports declarations, assignments, arithmetic expressions, and `return` only; no full C grammar, functions, branches, or loops.
- Synthetic data may not represent real programs, and similar patterns may occur in both train and test sets.
- The discriminator's score is not proof of semantic equivalence.
- This notebook does **not** implement a generator adversarial gradient estimator such as REINFORCE or Gumbel–Softmax. It provides supervised pretraining plus discriminator training as an initial experimental scaffold.
- The validator is intentionally basic. Differential execution over a finite set of examples is not a formal proof.
- Compare multiple seeds, increase dataset diversity, add control-flow support, and implement/document an adversarial update before drawing conclusions about GAN effectiveness.